# 社情民意周产出诊断（2026-08-24）

## TL;DR

本 notebook 只读分析 SQLite、滚动指标、配置和当天网页缓存。核心问题不是正式质量闸门过严，也不是当前 Token 上限不足，而是有效信源覆盖和候选前政策核验不足；同时现有漏斗日志不足以可靠测量规则误杀。

## Context & Methods

分析单位为自然周。成稿口径是已通过证据闸门并写入 `topics` 表的稿件；当前周尚未结束。当前来源产出按 2026-08-24 本地缓存重放，不能代表长期平均。

In [ ]:
from pathlib import Path
from urllib.parse import quote_plus
import hashlib
import json
import sqlite3
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
DB = ROOT / 'data/history/workflow.db'
METRICS = ROOT / 'outputs/review/metrics/novelty_rolling_21d.json'
assert DB.exists() and METRICS.exists()
conn = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)
conn.row_factory = sqlite3.Row
integrity = conn.execute('PRAGMA integrity_check').fetchone()[0]
assert integrity == 'ok', integrity
metrics = json.loads(METRICS.read_text(encoding='utf-8'))
print({'database_integrity': integrity, 'metrics_window_days': metrics['window_days']})

## Data

### 自然周产出漏斗

In [ ]:
weekly = pd.DataFrame(metrics['production_funnel']['weeks'])
weekly_view = weekly[[
    'week_start', 'is_closed_week', 'live_runs', 'candidate_count',
    'selected_topic_count', 'evidence_gate_passed_count',
    'approved_count', 'submitted_count', 'recorded_token_used'
]].copy()
weekly_view

In [ ]:
plot_rows = weekly[['week_start', 'candidate_count', 'evidence_gate_passed_count']].copy()
plot_rows['week_start'] = pd.to_datetime(plot_rows['week_start']).dt.strftime('%m-%d')
ax = plot_rows.set_index('week_start').plot(
    kind='bar', figsize=(8, 4), color=['#4C78A8', '#F58518'], width=0.72
)
ax.set_title('Weekly candidates and evidence-passed drafts')
ax.set_xlabel('Week starting')
ax.set_ylabel('Count')
ax.legend(['Candidates', 'Evidence-passed drafts'])
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 当天缓存中的来源有效面

这里重放每个已配置检索源的 90 天缓存结果，并按当前规则筛选。它能揭示当天的有效来源面，但不等于长期来源收益率。

In [ ]:
from sqmy.collector import SourceCollector
from sqmy.config import Settings
from sqmy.screener import rule_screen

settings = Settings.load(ROOT / 'config/settings.toml')
collector = SourceCollector(ROOT, settings.raw)
source_rows = []
all_items = []
for source in collector.sources:
    url = 'https://www.bing.com/news/search?q=' + quote_plus(source['query']) + '&format=rss&setlang=zh-cn'
    cache = ROOT / 'data/cache/feeds' / f"{hashlib.sha256(url.encode()).hexdigest()}.xml"
    payload = {'source': source}
    if cache.exists():
        payload['xml'] = cache.read_text(encoding='utf-8')
    items = collector._items_from_payloads([payload]) if cache.exists() else []
    qualified = rule_screen(items, settings.section('discovery'))
    source_rows.append({
        'source_id': source['id'],
        'tier': int(source.get('expansion_tier', 1)),
        'configured_level': int(source['level']),
        'raw_90d': len(items),
        'rule_qualified': len(qualified),
        'cached': cache.exists(),
    })
    all_items.extend(items)
source_yield = pd.DataFrame(source_rows).sort_values(
    ['raw_90d', 'rule_qualified', 'source_id'], ascending=[False, False, True]
)
source_yield

In [ ]:
source_summary = {
    'configured_sources': int(len(source_yield)),
    'zero_result_sources': int((source_yield.raw_90d == 0).sum()),
    'active_sources': int((source_yield.raw_90d > 0).sum()),
    'raw_items_with_cross_source_duplicates': int(source_yield.raw_90d.sum()),
    'globally_rule_qualified_after_dedup_cap': int(len(rule_screen(all_items, settings.section('discovery')))),
}
source_summary

### 制度新意反馈与 Token

In [ ]:
feedback = pd.read_sql_query(
    """
    SELECT review_outcome, COUNT(*) AS n
    FROM novelty_audits
    WHERE reviewed_at IS NOT NULL
      AND reviewed_at >= datetime('now', '-21 days')
    GROUP BY review_outcome ORDER BY n DESC, review_outcome
    """, conn
)
token_calls = pd.read_sql_query(
    """
    SELECT CASE WHEN m.created_at >= '2026-08-20T07:30:00+00:00'
                THEN 'post_isolation' ELSE 'pre_isolation' END AS period,
           COUNT(*) AS calls,
           SUM(m.input_tokens + m.output_tokens) AS tokens,
           ROUND(AVG(m.input_tokens + m.output_tokens), 0) AS avg_tokens,
           MIN(m.input_tokens + m.output_tokens) AS min_tokens,
           MAX(m.input_tokens + m.output_tokens) AS max_tokens,
           SUM(m.over_budget) AS over_budget_calls
    FROM model_calls m
    JOIN run_context rc ON rc.run_id = m.run_id
    WHERE rc.mode = 'live' AND m.task_id = 'screening'
    GROUP BY period ORDER BY period
    """, conn
)
display(feedback)
display(token_calls)

In [ ]:
data_quality = {
    'database_integrity': integrity,
    'live_runs': conn.execute("SELECT COUNT(*) FROM runs r JOIN run_context rc ON rc.run_id=r.id WHERE rc.mode='live'").fetchone()[0],
    'config_versions': conn.execute("SELECT COUNT(DISTINCT r.config_hash) FROM runs r JOIN run_context rc ON rc.run_id=r.id WHERE rc.mode='live'").fetchone()[0],
    'candidate_rows': conn.execute("SELECT COUNT(*) FROM candidates c JOIN run_context rc ON rc.run_id=c.run_id WHERE rc.mode='live'").fetchone()[0],
    'run_efficiency_candidate_total': conn.execute("SELECT COALESCE(SUM(e.candidate_count),0) FROM run_efficiency e JOIN run_context rc ON rc.run_id=e.run_id WHERE rc.mode='live'").fetchone()[0],
    'event_items_semantics': '仅持久化模型前候选池，不是全部采集结果',
    'selection_semantics': 'selected 为可变当前状态，不是不可变选择事件',
    'token_scope': '只含项目账本中的模型调用，不含未写入账本的交互式 Codex 深研和写作',
}
data_quality

## Results

1. **高置信度：主要瓶颈在候选前的信源和政策覆盖核验。** 最近 21 天 10 个有后续反馈的制度新意审查，结果为 `reversed` 4、`missed_coverage` 2、`reframed` 2、`stopped_other` 2，没有 `supported_gap`。这些标签含历史口径，不能直接当作精确错误率，但方向一致。
2. **高置信度：Token 预算不是当前首要瓶颈。** 上下文隔离后 3 次筛选调用为 19,892—25,273 Token，均低于 45,000 的阶段上限；单纯加预算不会补足原始政策证据。
3. **中高置信度：名义主题范围不小，实际获取面偏窄。** 当前配置 22 个来源，但当天缓存有 12 个来源为 0 结果；所有来源都依赖同一种 Bing News RSS 搜索方式。
4. **中等置信度：级联停止条件按数量而非质量，可能造成输入池构成偏差。** 达到 8 条新事件就停止扩展，不能保证其中包含足够的一级原始来源和可验证北京落点。
5. **无法确认：规则是否误杀了大量好题。** 当前只保存模型前池，不保存逐来源的采集、时间过滤、规则排除和历史排除明细；14 次真实运行还跨 9 个配置哈希，纵向比较受限。

## Takeaways

建议保持事实、证据和机制质量闸门不变，把严格性前移：先补可观测漏斗，再给前 8—12 条线索做零模型或小文本的原始来源定位与政策覆盖核验；按来源质量配额组成一次模型输入池；周一无合格题时增加一次只处理新增信息的低成本补扫。预算先保持不变，把原来可能重复的筛选调用额度转给定向核验。连续观察 2—3 周后，再决定是否扩源、改权重或提高预算。